# RBF SVM on frozen combined classifier features

Following the committed logistic feature comparisons, test an RBF-kernel SVM on **embeddings alone** and the four saved combinations: **diseases + criteria**, **diseases + pathways**, **criteria + pathways**, and **all three**. Use the exact saved matrices, preprocessing, DBSCAN pseudo-labels, and raw sample weights. No new features, preprocessing fit, or clustering.

Predeclared grid: **C=[1,10,100]**, **gamma=[0.001,"scale",0.03]**; 5 representations × 9 settings = **45 configurations**. Smaller gamma gives smoother, broader neighborhoods; larger gamma gives more local boundaries. `scale` derives gamma from the fitted training matrix variance.

Each reason is fitted independently with normalized fitting-weight total 157. SVM decisions are calibrated with sigmoid calibration using three stratified training-only folds, random_state=42, ensemble=False, and one final estimator trained on all included training rows for that reason. Native SVC probability estimation is not used. Keep the final cutoff at 0.5 and highest-probability fallback. Validation never enters calibration or fitting.

Compare against the 0.910256 sample-F1 logistic leader, the logistic model on the same representation, and an embeddings-only SVM with matching C/gamma. Save all models, predictions, reused matrices, and settings locally. No Kaggle submission, commit, or tag in this experiment.

In [1]:
from pathlib import Path
import ast
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits
from IPython.display import display

SEED=42
REASONS=["Safety","Funding","Efficacy","Recruitment","Other"]
EMBEDDINGS=[f"emb_{i}" for i in range(150)]
C_GRID=[1.0,10.0,100.0]
GAMMA_GRID=[0.001,"scale",0.03]
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/"part1",cwd/"AML-Midterm"/"part1"]
          if (p/"part1_train.csv").is_file() or (p/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR=DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
SOURCE=DATA/"experiments/classifier_combinations"
OUT=DATA/"experiments/nonlinear_combinations"
for folder in ["datasets","models","validation_predictions","submissions"]:(OUT/folder).mkdir(parents=True,exist_ok=True)
train=pd.read_csv(INPUT_DIR/"part1_train.csv")
valid=pd.read_csv(INPUT_DIR/"part1_val.csv")
truth=valid[REASONS].to_numpy(dtype=int)
original_y=train[REASONS].to_numpy(dtype=int)
known=train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert len(train)==1171 and len(valid)==78 and known.sum()==157
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isin(truth,[0,1]).all()
fixed=pd.read_csv(SOURCE/"datasets/fixed_targets_weights.csv",float_precision="round_trip")
assert fixed.ID.tolist()==train.ID.tolist()
Y=fixed[REASONS].to_numpy(dtype=int);W=fixed[[f"weight_{r}" for r in REASONS]].to_numpy()
np.testing.assert_array_equal(Y[known],original_y[known]);np.testing.assert_array_equal(W[known],np.ones((157,5)))
assert np.array_equal(W>0,Y!=-100)
(OUT/"datasets/fixed_targets_weights.csv").write_bytes((SOURCE/"datasets/fixed_targets_weights.csv").read_bytes())
source_settings=json.loads((SOURCE/"settings.json").read_text())
REPRESENTATIONS=list(source_settings["combinations"])
assert len(REPRESENTATIONS)==5 and REPRESENTATIONS[0]=="embeddings_only"
reference_sample=float(source_settings["reference_sample_f1"])
reference_macro=float(source_settings["reference_macro_f1"])
LOGISTIC_MODELS={rep:joblib.load(SOURCE/"models"/f"{rep}_weight1_C1000.joblib") for rep in REPRESENTATIONS}
source_comparison=pd.read_csv(SOURCE/"comparison.csv")
LOGISTIC_SCORES={rep:float(source_comparison.loc[(source_comparison.representation==rep)&(source_comparison.C==1000),"sample_f1"].iloc[0]) for rep in REPRESENTATIONS}
protected_paths=[INPUT_DIR/"part1_train.csv",INPUT_DIR/"part1_val.csv",SOURCE/"settings.json",SOURCE/"comparison.csv",SOURCE/"datasets/fixed_targets_weights.csv"]
MATRICES={}
for rep in REPRESENTATIONS:
    for suffix in [".npz","_columns.json"]:
        path=SOURCE/"datasets"/f"{rep}{suffix}"
        protected_paths.append(path);(OUT/"datasets"/path.name).write_bytes(path.read_bytes())
    protected_paths.append(SOURCE/"models"/f"{rep}_weight1_C1000.joblib")
    MATRICES[rep]=sparse.load_npz(OUT/"datasets"/f"{rep}.npz")
legacy_svm_path=DATA/"experiments/additional_methods/classifier_models/dbscan_m5_q25__crossover_rbf_svm.joblib"
legacy_svm=joblib.load(legacy_svm_path)
protected_paths.append(legacy_svm_path)
protected_hashes={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
print("45 RBF SVM settings; exact existing matrices, DBSCAN targets, and weights frozen.")
print("Earlier DBSCAN SVM parameters:",legacy_svm["parameters"])


45 RBF SVM settings; exact existing matrices, DBSCAN targets, and weights frozen.
Earlier DBSCAN SVM parameters: {'C': 10.0, 'gamma': 'scale'}


In [2]:
def disease_lists(frame):
    def parse(value):
        if pd.isna(value) or not str(value).strip():return []
        values=ast.literal_eval(value)
        assert isinstance(values,list)
        return sorted({str(v).strip().lower() for v in values if str(v).strip()})
    return frame.diseases_cleaned.map(parse).tolist()


def pathway_lists(frame):
    return frame.drug_pathways.fillna("").map(lambda s:sorted({v.strip() for v in str(s).split(";") if v.strip()})).tolist()


def missing_pathways(frame):
    return frame.drug_pathways.fillna("").astype(str).str.strip().eq("").to_numpy(dtype=float)[:,None]


def transform_block(block,frame):
    kind=block["kind"]
    if kind=="combination":return sparse.hstack([transform_block(component,frame) for component in block["components"]],format="csr")
    if kind=="none":return sparse.csr_matrix((len(frame),0))
    if kind=="missing_only":return sparse.csr_matrix(missing_pathways(frame))
    if kind=="criteria":return block["vectorizer"].transform(frame.criteria.fillna("")).tocsr()
    values=disease_lists(frame) if kind=="diseases" else pathway_lists(frame)
    allowed=set(block["encoder"].classes_)
    result=block["encoder"].transform([[v for v in row if v in allowed] for row in values]).astype(float).tocsr()
    if kind=="pathways":result=sparse.hstack([result,sparse.csr_matrix(missing_pathways(frame))],format="csr")
    assert result.shape[1]==len(block["feature_names"])
    assert np.isin(result.data,[0,1]).all()
    return result


def transform_model(model,frame):
    embeddings=model["embedding_scaler"].transform(frame[model["embedding_features"]].to_numpy())
    if model["block"]["kind"]=="none":return embeddings
    return sparse.hstack([sparse.csr_matrix(embeddings),transform_block(model["block"],frame)*model.get("feature_block_weight",1.0)],format="csr")


def predict_model(model,frame):
    matrix=transform_model(model,frame)
    return np.column_stack([est.predict_proba(matrix)[:,list(est.classes_).index(1)] for est in model["estimators"]])


def label_predictions(probabilities):
    assert np.isfinite(probabilities).all() and ((probabilities>=0)&(probabilities<=1)).all()
    predicted=(probabilities>=0.5).astype(int)
    empty=predicted.sum(axis=1)==0
    rows=np.flatnonzero(empty);predicted[rows,probabilities[rows].argmax(axis=1)]=1
    return predicted,empty

for rep in REPRESENTATIONS:
    expected=sparse.csr_matrix(transform_model(LOGISTIC_MODELS[rep],train))
    difference=MATRICES[rep]-expected
    assert not difference.nnz or abs(difference.data).max()<1e-12
    assert MATRICES[rep].shape[1]==150+len(LOGISTIC_MODELS[rep]["block"]["feature_names"])
reference_probability=predict_model(LOGISTIC_MODELS["embeddings_only"],valid)
reference_prediction,_=label_predictions(reference_probability)
np.testing.assert_allclose(f1_score(truth,reference_prediction,average="samples",zero_division=0),reference_sample,atol=1e-14)
reference_record={"sample_f1":reference_sample,"macro_f1":reference_macro}
ROWS=[];MATCHED_SVM_SCORES={};CALIBRATION_ROWS=[]


def parameter_key(C,gamma):return (float(C),str(gamma))


def fit_svm_candidate(rep,C,gamma):
    gamma_name=str(gamma).replace(".","p")
    name=f"{rep}_rbf_C{C:g}_gamma{gamma_name}"
    template=LOGISTIC_MODELS[rep]
    model={**template,"kind":"rbf_svm_feature_combinations","C":C,"gamma":gamma,"estimators":[],
           "training_matrix_file":f"datasets/{rep}.npz","source_targets_file":"datasets/fixed_targets_weights.csv",
           "source_logistic_model_file":str((SOURCE/"models"/f"{rep}_weight1_C1000.joblib").relative_to(DATA)),
           "calibration":{"method":"sigmoid","folds":3,"shuffle":True,"random_state":SEED,"ensemble":False}}
    matrix=MATRICES[rep].toarray() if rep=="embeddings_only" else MATRICES[rep]
    with threadpool_limits(limits=1):
        for j,r in enumerate(REASONS):
            included=W[:,j]>0;y=Y[included,j];weights=W[included,j]*157/W[included,j].sum()
            assert np.isin(y,[0,1]).all() and np.bincount(y,minlength=2).min()>=3
            splitter=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED)
            # Persist and verify the exact local-to-global calibration split mapping.
            global_indices=np.flatnonzero(included)
            split_hashes=[]
            for fold,(fit_rows,calibration_rows) in enumerate(splitter.split(matrix[included],y)):
                assert set(fit_rows).isdisjoint(calibration_rows)
                assert len(fit_rows)+len(calibration_rows)==included.sum()
                split_hashes.append(hashlib.sha256(global_indices[calibration_rows].tobytes()).hexdigest())
            estimator=CalibratedClassifierCV(
                estimator=SVC(kernel="rbf",C=C,gamma=gamma,random_state=SEED),
                method="sigmoid",cv=splitter,ensemble=False,n_jobs=1)
            with warnings.catch_warnings():
                warnings.simplefilter("error",ConvergenceWarning)
                estimator.fit(matrix[included],y,sample_weight=weights)
            model["estimators"].append(estimator)
            fitted=estimator.calibrated_classifiers_[0].estimator
            CALIBRATION_ROWS.append({"experiment":name,"reason":r,"training_rows":int(included.sum()),
                                     "known_rows":int((included&known).sum()),"generated_rows":int((included&~known).sum()),
                                     "normalized_weight_total":float(weights.sum()),"resolved_gamma":float(fitted._gamma),
                                     "support_vectors":int(fitted.n_support_.sum()),
                                     "fold_0_validation_global_rows_sha256":split_hashes[0],
                                     "fold_1_validation_global_rows_sha256":split_hashes[1],
                                     "fold_2_validation_global_rows_sha256":split_hashes[2]})
    probabilities=predict_model(model,valid)
    predicted,empty=label_predictions(probabilities)
    sample=f1_score(truth,predicted,average="samples",zero_division=0)
    macro=f1_score(truth,predicted,average="macro",zero_division=0)
    key=parameter_key(C,gamma)
    if rep=="embeddings_only":MATCHED_SVM_SCORES[key]=sample
    if rep=="embeddings_only" and C==legacy_svm["parameters"]["C"] and gamma==legacy_svm["parameters"]["gamma"]:
        scaled=legacy_svm["scaler"].transform(valid[EMBEDDINGS].to_numpy())
        previous=np.column_stack([e.predict_proba(scaled)[:,list(e.classes_).index(1)] for e in legacy_svm["estimators"]])
        np.testing.assert_allclose(probabilities,previous,rtol=1e-8,atol=1e-10)
    known_prediction,_=label_predictions(predict_model(model,train.loc[known]))
    ROWS.append({"experiment":name,"representation":rep,"C":C,"gamma":str(gamma),
                 "additional_features":len(template["block"]["feature_names"]),"total_features":matrix.shape[1],
                 "sample_f1":sample,"macro_f1":macro,
                 "delta_vs_current_leader":sample-reference_sample,
                 "delta_vs_same_representation_logistic_C1000":sample-LOGISTIC_SCORES[rep],
                 "delta_vs_matched_embedding_svm":sample-MATCHED_SVM_SCORES[key],
                 "known_training_sample_f1":f1_score(original_y[known],known_prediction,average="samples",zero_division=0),
                 "fallback_trials":int(empty.sum()),"changed_trials_vs_current":int((predicted!=reference_prediction).any(axis=1).sum()),
                 **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0))),"kaggle_score":np.nan})
    frame=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):frame[f"probability_{r}"]=probabilities[:,j];frame[f"predicted_{r}"]=predicted[:,j]
    frame["used_fallback"]=empty
    frame["trial_f1"]=2*(truth*predicted).sum(axis=1)/(truth.sum(axis=1)+predicted.sum(axis=1))
    frame.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    joblib.dump(model,OUT/"models"/f"{name}.joblib",compress=3)
    pd.DataFrame(ROWS).to_csv(OUT/"comparison.csv",index=False)
    pd.DataFrame(CALIBRATION_ROWS).to_csv(OUT/"calibration_audit.csv",index=False)
    return ROWS[-1]


def run_representation(rep):
    for C in C_GRID:
        for gamma in GAMMA_GRID:fit_svm_candidate(rep,C,gamma)
    subset=pd.DataFrame(ROWS).query("representation==@rep")
    best=subset.sort_values(["sample_f1","C","gamma"],ascending=[False,True,True],kind="stable").iloc[0]
    print(f"{rep}: {len(subset)} SVM settings; best sample F1={best.sample_f1:.6f}, macro F1={best.macro_f1:.6f}, "
          f"C={best.C:g}, gamma={best.gamma}, change vs logistic leader={best.delta_vs_current_leader:+.6f}",flush=True)

run_representation("embeddings_only")


embeddings_only: 9 SVM settings; best sample F1=0.867521, macro F1=0.651287, C=100, gamma=0.001, change vs logistic leader=-0.042735


In [3]:
run_representation("diseases_criteria")


diseases_criteria: 9 SVM settings; best sample F1=0.858974, macro F1=0.712807, C=1, gamma=scale, change vs logistic leader=-0.051282


In [4]:
run_representation("diseases_pathways")


diseases_pathways: 9 SVM settings; best sample F1=0.858974, macro F1=0.712807, C=1, gamma=scale, change vs logistic leader=-0.051282


In [5]:
run_representation("criteria_pathways")


criteria_pathways: 9 SVM settings; best sample F1=0.858974, macro F1=0.712807, C=1, gamma=scale, change vs logistic leader=-0.051282


In [6]:
run_representation("diseases_criteria_pathways")


diseases_criteria_pathways: 9 SVM settings; best sample F1=0.858974, macro F1=0.712807, C=1, gamma=scale, change vs logistic leader=-0.051282


In [7]:
comparison=pd.DataFrame(ROWS);assert len(comparison)==45
ranked=comparison.sort_values(["sample_f1","additional_features","C","gamma","experiment"],ascending=[False,True,True,True,True],kind="stable")
best=ranked.iloc[0]
best_by_representation=ranked.groupby("representation",sort=False).head(1)
best_by_representation.to_csv(OUT/"best_by_representation.csv",index=False)
comparison.query("representation=='embeddings_only'").to_csv(OUT/"embedding_controls.csv",index=False)
display(best_by_representation[["representation","C","gamma","sample_f1","macro_f1","delta_vs_current_leader"]].round(6))
selected=joblib.load(OUT/"models"/f"{best.experiment}.joblib")
selected_probability=predict_model(selected,valid)
selected_prediction,_=label_predictions(selected_probability)
# Reload every saved model and independently rescore its saved label set.
for row in comparison.itertuples(index=False):
    frame=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==valid.ID.tolist()
    probabilities=frame[[f"probability_{r}" for r in REASONS]].to_numpy()
    predictions=frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    np.testing.assert_array_equal(label_predictions(probabilities)[0],predictions)
    np.testing.assert_allclose(f1_score(truth,predictions,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,predictions,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
    loaded=joblib.load(OUT/"models"/f"{row.experiment}.joblib")
    np.testing.assert_allclose(predict_model(loaded,valid),probabilities,rtol=1e-12,atol=1e-12)
# Independently refit the selected calibrated SVM from saved matrices and target weights.
replayed_fixed=pd.read_csv(OUT/"datasets/fixed_targets_weights.csv",float_precision="round_trip")
assert replayed_fixed.ID.tolist()==train.ID.tolist()
np.testing.assert_array_equal(replayed_fixed[REASONS].to_numpy(dtype=int),Y)
np.testing.assert_array_equal(replayed_fixed[[f"weight_{r}" for r in REASONS]].to_numpy(),W)
matrix=sparse.load_npz(OUT/selected["training_matrix_file"])
expected=sparse.csr_matrix(transform_model(selected,train));difference=matrix-expected
assert not difference.nnz or abs(difference.data).max()<1e-12
if selected["block"]["kind"]=="none":matrix=matrix.toarray()
replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j in range(5):
        included=W[:,j]>0;estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(matrix[included],Y[included,j],sample_weight=W[included,j]*157/W[included,j].sum())
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_model(replay,valid),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_model(replay,valid))[0],selected_prediction)
for rel,digest in protected_hashes.items():assert hashlib.sha256((DATA/rel).read_bytes()).hexdigest()==digest
print("All 45 saved models reload and rescore; original DBSCAN SVM reproduced; winner independent calibrated refit and protected hashes passed.")


,representation,C,gamma,sample_f1,macro_f1,delta_vs_current_leader
6,embeddings_only,100.0,0.001,0.867521,0.651287,-0.042735
19,diseases_pathways,1.0,scale,0.858974,0.712807,-0.051282
28,criteria_pathways,1.0,scale,0.858974,0.712807,-0.051282
10,diseases_criteria,1.0,scale,0.858974,0.712807,-0.051282
37,diseases_criteria_pathways,1.0,scale,0.858974,0.712807,-0.051282


All 45 saved models reload and rescore; original DBSCAN SVM reproduced; winner independent calibrated refit and protected hashes passed.


In [8]:
candidate=None
if best.sample_f1>reference_sample+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    prediction,empty=label_predictions(predict_model(selected,test))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_dbscan_{best.experiment}_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"classifier":"RBF SVM","C_grid":C_GRID,"gamma_grid":GAMMA_GRID,"representations":REPRESENTATIONS,"configurations":45,
          "calibration":{"method":"sigmoid","folds":3,"shuffle":True,"random_state":SEED,"ensemble":False,"training_only":True},
          "source_experiment":"classifier_combinations","reference_sample_f1":reference_sample,"reference_macro_f1":reference_macro,
          "generated_labels_changed":False,"generated_weights_changed":False,"preprocessing_refitted":False,
          "feature_block_weights":1.0,"embedding_scaler_fit_rows":157,"original_vocabulary_idf_fit_rows":1171,
          "validation_rows":78,"validation_used_for_fit":False,"fitting_weight_total_per_reason":157,"cutoff":0.5,"fallback":True,
          "best_svm_experiment":best.experiment,"best_svm_sample_f1":float(best.sample_f1),"best_svm_macro_f1":float(best.macro_f1),
          "leader_changed":bool(best.sample_f1>reference_sample+1e-12),"candidate":candidate,
          "protected_input_sha256":protected_hashes,
          "dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*')) if p.is_file()},
          "selection":"sample F1; ties prefer fewer added features, then smaller C, then gamma string",
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"scipy":scipy.__version__,"numpy":np.__version__,"pandas":pd.__version__},
          "new_kaggle_scores":None,"reported_best_kaggle":0.87393}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
def table(frame):
    lines=["| Representation | C | Gamma | Sample F1 | Macro F1 | Change vs logistic leader | Change vs matched embedding SVM |",
           "|---|---:|---|---:|---:|---:|---:|"]
    for row in frame.itertuples(index=False):
        lines.append(f"| {row.representation} | {row.C:g} | {row.gamma} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.delta_vs_current_leader:+.6f} | {row.delta_vs_matched_embedding_svm:+.6f} |")
    return "\n".join(lines)
lines=["# RBF SVM on frozen feature combinations","",
       "45 settings: C=1,10,100; gamma=0.001,scale,0.03; embeddings alone and all four saved pair/triple combinations. "
       "Inputs, vocabularies, IDF, embedding scaler, DBSCAN targets, and raw weights are copied unchanged from the logistic experiment. "
       "Each reason uses normalized fitting-weight total 157. Embeddings-only uses dense input as in the earlier SVM check; combinations use sparse matrices.","",
       "Probabilities use sigmoid calibration with three stratified training-only folds (seed 42, ensemble=False). "
       "The final estimator is trained on all included training rows for each reason. Calibration audit records support-vector counts, "
       "resolved gamma, training-row counts, and hashes of each held-out training-fold row mapping. "
       "The 78 external validation trials never enter fitting or calibration. Final cutoff=0.5, highest-probability fallback enabled.","",
       f"Current logistic leader: sample F1 **{reference_sample:.6f}**, macro F1 **{reference_macro:.6f}**. "
       f"Best SVM: **{best.experiment}**, sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "## Best SVM by representation","",table(best_by_representation),"",
       "## All configurations","",table(ranked),"",
       "## Verification and interpretation","",
       "All 45 saved models were reloaded and their probabilities/predictions independently checked and rescored. "
       "The earlier DBSCAN SVM control is reproduced when C/gamma match. The best SVM independently refits from the saved "
       "matrix and fixed targets/weights with matching calibrated probabilities; protected original artifacts retain their hashes.","",
       "comparison.csv reports changes against the current logistic leader, the C=1000 logistic model on the same input, "
       "and the embeddings-only SVM at matching C/gamma. Thus gains from new input groups can be distinguished from a classifier change. "
       "These are validation-selected alternatives, not independent test estimates or a proof that every nonlinear model will behave similarly.","",
       "datasets/ preserves exact reused matrices and ordered columns with the shared fixed targets/weights; models/ saves "
       "native sklearn preprocessing and calibrated classifiers. validation_predictions/, comparison.csv, best_by_representation.csv, "
       "embedding_controls.csv, calibration_audit.csv, and settings.json preserve results and checksums.","",
       "No Kaggle submission, commit, or tag was made after this experiment. The preceding logistic feature work was committed before this run.",""]
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No SVM configuration improves the primary validation metric; retain the current logistic leader.")
(OUT/"SUMMARY.md").write_text("\n".join(lines)+"\n")
print(table(best_by_representation))
print("Leader changed:",settings["leader_changed"],"Candidate:",candidate)


| Representation | C | Gamma | Sample F1 | Macro F1 | Change vs logistic leader | Change vs matched embedding SVM |
|---|---:|---|---:|---:|---:|---:|
| embeddings_only | 100 | 0.001 | 0.867521 | 0.651287 | -0.042735 | +0.000000 |
| diseases_pathways | 1 | scale | 0.858974 | 0.712807 | -0.051282 | +0.000000 |
| criteria_pathways | 1 | scale | 0.858974 | 0.712807 | -0.051282 | +0.000000 |
| diseases_criteria | 1 | scale | 0.858974 | 0.712807 | -0.051282 | +0.000000 |
| diseases_criteria_pathways | 1 | scale | 0.858974 | 0.712807 | -0.051282 | +0.000000 |
Leader changed: False Candidate: None
